In [1]:
import sys
import os
import glob
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px

sys.path.append('..')
from src.database import init_db, get_connection
from src.processor import procesar_excel_brou

init_db()

# Buscar archivo
files = glob.glob("../data/raw/*.xls*")
if not files:
    print("❌ No se encontraron archivos.")
else:
    latest_file = max(files, key=os.path.getctime)
    df_nuevo = procesar_excel_brou(latest_file)
    # Creamos la columna categoría vacía para obligar a llenarla
    if 'categoria' not in df_nuevo.columns:
        df_nuevo['categoria'] = None
    print(f"✅ Archivo cargado. {len(df_nuevo)} movimientos detectados.")

--- DEBUG: Iniciando lectura de ../data/raw/Detalle_Movimiento_Cuenta (1).xls ---
--- DEBUG: ¡Tabla real encontrada en fila 31! ---
--- DEBUG: Columnas finales detectadas: ['Fecha', 'Descripción', 'Unnamed: 2', 'Número de documento', 'Asunto', 'Dependencia', 'Débito', 'Crédito']
--- DEBUG: Se procesaron 21 movimientos ---
✅ Archivo cargado. 21 movimientos detectados.


In [4]:
CATEGORIAS = ["Super", "Fijos", "Ocio", "Transporte", "Salud", "Educación", "Delivery", "Sueldo", "Otros", "-- BUG --"]

def procesar_categorias():
    clear_output()
    # Buscamos los que aún no tienen categoría
    pendientes = df_nuevo[df_nuevo['categoria'].isnull()]
    
    if not pendientes.empty:
        idx = pendientes.index[0]
        row = pendientes.iloc[0]
        
        print(f"--- CATEGORIZACIÓN OBLIGATORIA ({len(pendientes)} restantes) ---")
        print(f"Detalle: {row['descripcion']}")
        print(f"Monto: ${row['monto']}")
        
        selector = widgets.Dropdown(options=CATEGORIAS, description='Categoría:')
        boton = widgets.Button(description="Confirmar", button_style='success')
        
        def al_confirmar(b):
            df_nuevo.at[idx, 'categoria'] = selector.value
            procesar_categorias() # Salta al siguiente
            
        boton.on_click(al_confirmar)
        display(selector, boton)
    else:
        print("✅ ¡Todo categorizado! Ya puedes ejecutar la siguiente celda para guardar y graficar.")

procesar_categorias()

Dropdown(description='Categoría:', options=('Super', 'Fijos', 'Ocio', 'Transporte', 'Salud', 'Educación', 'Del…

Button(button_style='success', description='Confirmar', style=ButtonStyle())

In [3]:
if df_nuevo['categoria'].isnull().any():
    print("❌ Error: Todavía hay movimientos sin categoría. Completa la celda anterior.")
else:
    conn = get_connection()
    try:
        # Limpiamos para evitar errores de duplicados y asegurar datos nuevos
        conn.execute("DELETE FROM movimientos")
        df_nuevo.to_sql("movimientos", conn, if_exists="append", index=False, method="multi")
        conn.commit()
        print("💾 Datos guardados en la base de datos.")
        
        # Gráfico
        df_total = pd.read_sql("SELECT * FROM movimientos", conn)
        df_total['monto'] = pd.to_numeric(df_total['monto'])
        df_gastos = df_total[df_total['monto'] < 0].copy()
        df_gastos['monto_abs'] = df_gastos['monto'].abs()
        
        fig = px.pie(
            df_gastos, 
            values='monto_abs', 
            names='categoria', 
            title='Resumen de Gastos por Categoría',
            hole=0.4,
            template="plotly_dark"
        )
        fig.show()
        
    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        conn.close()

❌ Error: Todavía hay movimientos sin categoría. Completa la celda anterior.
